# Day 54 · Scale it up a little and count the cost

**First-principles question:** What does it cost to make a model bigger?

The full task (learn → build → record) is in this folder's `README.md`. This notebook is your workbench:
run cells top to bottom with **Shift + Enter**, fill in every cell marked **TODO**, and finish with the
**✅ self-check** cell at the end.

**Where to run it:** on your Mac (`device` becomes `mps` automatically), or upload this notebook to
[Google Colab](https://colab.research.google.com) and pick a free GPU (Runtime → Change runtime type). In Colab,
replace the first code cell with the `ROOT = Path.cwd()` line shown in its comment.

In [ ]:
import sys
from pathlib import Path

def find_root():
    """The fde-first-principles folder (it has progress.json)."""
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / "progress.json").exists():
            return p
    raise RuntimeError("Open this notebook from inside your fde-first-principles folder.")

ROOT = find_root()
sys.path.insert(0, str(ROOT))  # so `import lab.…` works

import urllib.request
import torch

def download(url, name):
    path = ROOT / "data" / "downloads" / name
    if not path.exists():
        path.parent.mkdir(parents=True, exist_ok=True)
        urllib.request.urlretrieve(url, path)
    return path.read_text(encoding="utf-8")

text = download("https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt",
                "tinyshakespeare.txt")
chars = sorted(set(text))
vocab_size = len(chars)
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}
encode = lambda s: [stoi[c] for c in s]
decode = lambda ids: "".join(itos[i] for i in ids)
data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9 * len(data))
train_data, val_data = data[:n], data[n:]
device = "mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(1337)
print(f"{len(text):,} characters · vocabulary {vocab_size} · device {device}")

def get_batch(split, batch_size, block_size):
    d = train_data if split == "train" else val_data
    ix = torch.randint(len(d) - block_size, (batch_size,))
    x = torch.stack([d[i:i + block_size] for i in ix])
    y = torch.stack([d[i + 1:i + block_size + 1] for i in ix])
    return x.to(device), y.to(device)
import time
import torch.nn as nn
from torch.nn import functional as F
# In Google Colab: delete the lines above find_root() … sys.path, and set ROOT = Path.cwd()

## 1 · A bigger model
Change these numbers (Karpathy's full size: BLOCK 256, N_EMBD 384, N_HEAD 6, N_LAYER 6 — only on a GPU).

In [ ]:
BATCH, BLOCK, N_EMBD, N_HEAD, N_LAYER = 32, 128, 128, 4, 4
MINUTES, LR = 10, 3e-4

class Head(nn.Module):
    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(N_EMBD, head_size, bias=False)
        self.query = nn.Linear(N_EMBD, head_size, bias=False)
        self.value = nn.Linear(N_EMBD, head_size, bias=False)
        self.register_buffer("tril", torch.tril(torch.ones(BLOCK, BLOCK)))

    def forward(self, x):
        B, T, C = x.shape
        k, q = self.key(x), self.query(x)
        wei = q @ k.transpose(-2, -1) * k.shape[-1] ** -0.5
        wei = F.softmax(wei.masked_fill(self.tril[:T, :T] == 0, float("-inf")), dim=-1)
        return wei @ self.value(x)

class Block(nn.Module):
    def __init__(self):
        super().__init__()
        hs = N_EMBD // N_HEAD
        self.heads = nn.ModuleList([Head(hs) for _ in range(N_HEAD)])
        self.proj = nn.Linear(N_EMBD, N_EMBD)
        self.ff = nn.Sequential(nn.Linear(N_EMBD, 4 * N_EMBD), nn.ReLU(), nn.Linear(4 * N_EMBD, N_EMBD))
        self.ln1, self.ln2 = nn.LayerNorm(N_EMBD), nn.LayerNorm(N_EMBD)

    def forward(self, x):
        h = self.ln1(x)
        x = x + self.proj(torch.cat([head(h) for head in self.heads], dim=-1))
        return x + self.ff(self.ln2(x))

class TinyGPT(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, N_EMBD)
        self.position_embedding = nn.Embedding(BLOCK, N_EMBD)
        self.blocks = nn.Sequential(*[Block() for _ in range(N_LAYER)])
        self.ln_f = nn.LayerNorm(N_EMBD)
        self.lm_head = nn.Linear(N_EMBD, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        x = self.token_embedding(idx) + self.position_embedding(torch.arange(T, device=idx.device))
        logits = self.lm_head(self.ln_f(self.blocks(x)))
        loss = None if targets is None else F.cross_entropy(logits.view(B * T, -1), targets.view(B * T))
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, n_new):
        for _ in range(n_new):
            logits, _ = self(idx[:, -BLOCK:])
            idx = torch.cat([idx, torch.multinomial(F.softmax(logits[:, -1, :], dim=-1), 1)], dim=1)
        return idx

model = TinyGPT().to(device)

## 2 · Count the parameters by hand
A Linear(a → b) has a·b weights + b biases. An Embedding(n, d) has n·d. A LayerNorm(d) has 2·d.

In [ ]:
def count_by_formula():
    linear = lambda a, b, bias=True: a * b + (b if bias else 0)
    # TODO: add up every part of the model:
    #   token_embedding (vocab_size × N_EMBD) + position_embedding (BLOCK × N_EMBD)
    #   per block: N_HEAD heads × 3 linears without bias (N_EMBD → head size)
    #              + proj (N_EMBD → N_EMBD) + ff (N_EMBD → 4·N_EMBD → N_EMBD) + 2 LayerNorms
    #   ln_f (a LayerNorm) + lm_head (N_EMBD → vocab_size)
    ...

by_pytorch = sum(p.numel() for p in model.parameters())
print(f"formula {count_by_formula():,} · pytorch {by_pytorch:,}")

## 3 · Train for a fixed time

In [ ]:
start = torch.zeros((1, 1), dtype=torch.long, device=device)
before = decode(model.generate(start, 200)[0].tolist())
optimizer = torch.optim.AdamW(model.parameters(), lr=LR)
log, t0, step = [], time.perf_counter(), 0
while time.perf_counter() - t0 < MINUTES * 60:
    xb, yb = get_batch("train", BATCH, BLOCK)
    _, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True); loss.backward(); optimizer.step()
    if step % 100 == 0:
        with torch.no_grad():
            val = model(*get_batch("val", BATCH, BLOCK))[1].item()
        log.append((step, loss.item(), val)); print(step, round(loss.item(), 3), round(val, 3))
    step += 1
seconds = time.perf_counter() - t0
print(f"{step} steps in {seconds / 60:.1f} minutes on {device}")
print("BEFORE:\n", before, "\nAFTER:\n", decode(model.generate(start, 300)[0].tolist()))

## 4 · What did it cost?
Look up one hourly price for a rented GPU. Then: what would 10× the parameters cost to train for the same number of tokens?

In [ ]:
# TODO: the hourly price you found (in ₹) and your estimate
GPU_PRICE_PER_HOUR_INR = None
cost_this_run = None             # GPU_PRICE_PER_HOUR_INR × hours (if you ran on a GPU)
print(cost_this_run)

In [ ]:
# ✅ Self-check: run me last. A clear message tells you what to fix.
assert count_by_formula() == by_pytorch, "Your formula must match PyTorch's count"
assert len(log) >= 2 and log[-1][2] < log[0][2], "The validation loss should go down"
assert GPU_PRICE_PER_HOUR_INR is not None, "Step 4: look up a price"
print("All checks passed ✅  Now write 'In my words' in README.md and record your video.")